<a href="https://colab.research.google.com/github/simjonghyeon04/-/blob/main/Fast_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fastapi uvicorn pydantic gradio httpx beautifulsoup4 deep-translator pandas pyngrok matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.6 MB/s eta 0:00:00


In [4]:
import os
import re
import sqlite3
from collections import Counter
import asyncio
from threading import Thread
import time

import httpx
import pandas as pd
from bs4 import BeautifulSoup
from deep_translator import GoogleTranslator
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel
import gradio as gr
import nest_asyncio

nest_asyncio.apply()

# ==========================================
# 1. 데이터베이스(SQLite3) 초기화 및 설정
# ==========================================
DB_PATH = "quotes_system.db"

def init_db():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS quotes (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            text TEXT NOT NULL,
            author TEXT NOT NULL,
            tags TEXT
        )
    """)
    conn.commit()
    conn.close()

init_db()

# ==========================================
# 2. FastAPI 백엔드 및 Pydantic 모델 설계
# ==========================================
app = FastAPI(title="격언 관리 및 데이터 분석 시스템", version="1.0")

class QuoteCreate(BaseModel):
    text: str
    author: str
    tags: str = ""

class QuoteUpdate(BaseModel):
    text: str
    author: str
    tags: str = ""

# --- RESTful API 엔드포인트 (CRUD) ---

@app.get("/quotes")
def get_all_quotes():
    """전체 격언 목록 조회"""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT id, text, author, tags FROM quotes")
    rows = cursor.fetchall()
    conn.close()

    return [{"id": r[0], "text": r[1], "author": r[2], "tags": r[3]} for r in rows]

@app.post("/quotes", status_code=status.HTTP_201_CREATED)
def create_quote(quote: QuoteCreate):
    """신규 격언 등록"""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO quotes (text, author, tags) VALUES (?, ?, ?)",
        (quote.text, quote.author, quote.tags)
    )
    conn.commit()
    new_id = cursor.lastrowid
    conn.close()
    return {"message": "격언이 성공적으로 등록되었습니다.", "id": new_id}

@app.put("/quotes/{qid}")
def update_quote(qid: int, quote: QuoteUpdate):
    """특정 ID의 격언 내용 수정 (핵심 요구사항)"""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT id FROM quotes WHERE id = ?", (qid,))
    if not cursor.fetchone():
        conn.close()
        raise HTTPException(status_code=404, detail="해당 ID의 격언을 찾을 수 없습니다.")

    cursor.execute(
        "UPDATE quotes SET text = ?, author = ?, tags = ? WHERE id = ?",
        (quote.text, quote.author, quote.tags, qid)
    )
    conn.commit()
    conn.close()
    return {"message": f"ID {qid}번 격언이 성공적으로 수정되었습니다."}

@app.delete("/quotes/{qid}")
def delete_quote(qid: int):
    """특정 격언 삭제"""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT id FROM quotes WHERE id = ?", (qid,))
    if not cursor.fetchone():
        conn.close()
        raise HTTPException(status_code=404, detail="해당 ID의 격언을 찾을 수 없습니다.")

    cursor.execute("DELETE FROM quotes WHERE id = ?", (qid,))
    conn.commit()
    conn.close()
    return {"message": f"ID {qid}번 격언이 성공적으로 삭제되었습니다."}

# ==========================================
# 3. 비즈니스 로직 및 크롤링 파이프라인
# ==========================================
def fetch_and_crawl_quotes():
    """외부 웹사이트에서 격언 20개를 수집하여 DB에 저장"""
    url = "https://quotes.toscrape.com"
    quotes_collected = []
    page = 1

    # 격언 20개를 채울 때까지 웹 브라우징 동적 크롤링 파이프라인
    with httpx.Client() as client:
        while len(quotes_collected) < 20:
            response = client.get(f"{url}/page/{page}/")
            if response.status_code != 200:
                break

            soup = BeautifulSoup(response.text, "html.parser")
            quote_elements = soup.find_all("div", class_="quote")

            if not quote_elements:
                break

            for elem in quote_elements:
                if len(quotes_collected) >= 20:
                    break
                text = elem.find("span", class_="text").text.replace("“", "").replace("”", "")
                author = elem.find("small", class_="author").text
                tags = [t.text for t in elem.find_all("a", class_="tag")]
                tags_str = ", ".join(tags)

                quotes_collected.append((text, author, tags_str))
            page += 1

    # 데이터베이스 적재
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.executemany(
        "INSERT INTO quotes (text, author, tags) VALUES (?, ?, ?)",
        quotes_collected
    )
    conn.commit()
    conn.close()
    return f"성공적으로 20개의 격언을 수집하여 DB에 저장했습니다!"

# ==========================================
# 4. 프론트엔드 대시보드 (Gradio UI 데이터 연동)
# ==========================================
def get_gradio_table():
    """DB 내부 데이터를 불러와 Gradio 데이터프레임 양식으로 전송"""
    data = get_all_quotes()
    if not data:
        return pd.DataFrame(columns=["ID", "격언(영문)", "저자", "태그"])
    return pd.DataFrame(data).rename(columns={"id": "ID", "text": "격언(영문)", "author": "저자", "tags": "태그"})

def search_quotes(query):
    """실시간 검색: 저자 및 내용 기반 필터링"""
    df = get_gradio_table()
    if df.empty:
        return df
    filtered_df = df[df["격언(영문)"].str.contains(query, case=False) | df["저자"].str.contains(query, case=False)]
    return filtered_df

def translate_selected_text(text):
    """실시간 번역: deep-translator 연동"""
    if not text:
        return "번역할 격언을 선택하거나 입력해 주세요."
    try:
        translated = GoogleTranslator(source='en', target='ko').translate(text)
        return translated
    except Exception as e:
        return f"번역 오류: {str(e)}"

def generate_insights():
    """데이터 시각화: 단어 빈도수 차트 데이터 생성"""
    df = get_gradio_table()
    if df.empty or len(df) < 1:
        return gr.BarPlot(), gr.BarPlot()

    # 1. 단어 빈도수 분석 (불용어 제외)
    all_text = " ".join(df["격언(영문)"].tolist()).lower()
    words = re.findall(r'\b\w+\b', all_text)
    stopwords = {'the', 'a', 'to', 'and', 'of', 'in', 'is', 'that', 'it', 'on', 'you', 'i', 'his', 'her', 'he', 'she', 'for', 'with', 'as', 'was' }
    filtered_words = [w for w in words if w not in stopwords and len(w) > 2]

    word_counts = Counter(filtered_words).most_common(10)
    word_df = pd.DataFrame(word_counts, columns=["Word", "Frequency"])

    # 2. 저자별 점유율 분포
    author_counts = df["저자"].value_counts().reset_index()
    author_counts.columns = ["Author", "Count"]

    # Gradio 전용 컴포넌트인 gr.BarPlot 포맷에 맞추어 반환
    word_chart = gr.BarPlot(word_df, x="Word", y="Frequency", title="가장 많이 사용된 단어 Top 10", vertical=False, width=400)
    author_chart = gr.BarPlot(author_counts, x="Author", y="Count", title="저자별 격언 점유 분포", width=400)

    return word_chart, author_chart

# --- Gradio 레이아웃 인터페이스 구축 ---
with gr.Blocks(title="격언 관리 대시보드") as dashboard:
    gr.Markdown("# 📜 격언 관리 및 실시간 데이터 분석 대시보드")

    with gr.Tab("데이터 조회 및 동기화"):
        with gr.Row():
            crawl_btn = gr.Button("🌐 외부 격언 수집 파이프라인 가동 (20개)", variant="primary")
            refresh_btn = gr.Button("🔄 대시보드 데이터 갱신")

        status_output = gr.Markdown()
        search_box = gr.Textbox(label="🔍 실시간 검색 (저자 또는 내용 입력)", placeholder="검색어를 입력하면 실시간으로 필터링됩니다.")

        data_table = gr.Dataframe(value=get_gradio_table(), interactive=False)

        crawl_btn.click(fn=fetch_and_crawl_quotes, outputs=status_output).then(fn=get_gradio_table, outputs=data_table)
        refresh_btn.click(fn=get_gradio_table, outputs=data_table)
        search_box.change(fn=search_quotes, inputs=search_box, outputs=data_table)

    with gr.Tab("실시간 인공지능 번역"):
        gr.Markdown("### 🔤 영어 격언 실시간 한글 번역기")
        input_quote = gr.Textbox(label="영어 격언 문장 (위 테이블에서 복사하거나 직접 입력하세요)", lines=3)
        translate_btn = gr.Button("번역하기", variant="secondary")
        output_translation = gr.Textbox(label="한국어 번역 결과", lines=3, interactive=False)

        translate_btn.click(fn=translate_selected_text, inputs=input_quote, outputs=output_translation)

    with gr.Tab("📊 시각화 데이터 인사이트"):
        analyze_btn = gr.Button("차트 데이터 분석 및 시각화 활성화", variant="primary")
        with gr.Row():
            word_barplot = gr.BarPlot()
            author_barplot = gr.BarPlot()

        analyze_btn.click(fn=generate_insights, outputs=[word_barplot, author_barplot])

# ① 서비스 통합: FastAPI 애플리케이션에 Gradio 마운트하기
app = gr.mount_gradio_app(app, dashboard, path="/ui")

# ==========================================
# 5. 로컬 서버 테스트 구동용 실행부
# ==========================================
if __name__ == "__main__":
    import uvicorn

    def run_uvicorn_app():
        """Function to run the uvicorn server."""
        uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

    # Run uvicorn in a separate thread to not block the Colab kernel
    uvicorn_thread = Thread(target=run_uvicorn_app)
    uvicorn_thread.start()

    print("FastAPI app is running in a background thread. Access Gradio UI at /ui.")
    print("If you need to stop the server, interrupt the kernel.")

    # Give the server a moment to start up
    time.sleep(5)


/usr/local/lib/python3.12/dist-packages/IPython/core/inputtransformer2.py:485: RuntimeWarning: coroutine 'Server.serve' was never awaited
  tokens_by_line.append([])


new /ui
FastAPI app is running in a background thread. Access Gradio UI at /ui.
If you need to stop the server, interrupt the kernel.


INFO:     Started server process [703]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [9]:
from pyngrok import ngrok

# 1. 본인의 ngrok Authtoken이 있다면 설정 (없어도 기본 구동은 가능하나 세션 제한이 있을 수 있음)
# IMPORTANT: Replace YOUR_NGROK_AUTH_TOKEN with your actual ngrok authtoken after signing up at https://dashboard.ngrok.com/
!ngrok config add-authtoken 3DNRU3mtymTM6aQiSHAvGuZShFu_5GpJRtjsk3jytkqDtoyHm

# 2. 8000번 포트로 포워딩 오픈
public_url = ngrok.connect(8000)
print(f"🔗 [API Swagger 명세서 문서]: {public_url.public_url}/docs")
print(f"🔗 [사용자 Gradio UI 대시보드]: {public_url.public_url}/ui")

# The uvicorn server is already running in a background thread from the previous cell, so this part is removed.

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
🔗 [API Swagger 명세서 문서]: https://clamp-buckwheat-sardine.ngrok-free.dev/docs
🔗 [사용자 Gradio UI 대시보드]: https://clamp-buckwheat-sardine.ngrok-free.dev/ui


In [7]:
import httpx

try:
    # Attempt to connect to the FastAPI server's /quotes endpoint
    response = httpx.get("http://127.0.0.1:8000/quotes")
    response.raise_for_status()  # Raise an exception for bad status codes (4xx or 5xx)
    print("FastAPI server is running and reachable!")
    print("Response from /quotes endpoint:")
    print(response.json())
except httpx.RequestError as e:
    print(f"FastAPI server is not reachable: {e}")
except httpx.HTTPStatusError as e:
    print(f"FastAPI server returned an error status: {e.response.status_code} - {e.response.text}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

INFO:     127.0.0.1:45804 - "GET /quotes HTTP/1.1" 200 OK
FastAPI server is running and reachable!
Response from /quotes endpoint:
[]


In [8]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_PATH)
df_quotes = pd.read_sql_query("SELECT * FROM quotes", conn)
conn.close()

if df_quotes.empty:
    print("The 'quotes' table is currently empty.")
else:
    print("Content of the 'quotes' table:")
    display(df_quotes)

The 'quotes' table is currently empty.
